# Фінальне домашнє завдання — AI-система збагачення Excel-даних

**Мета:** створити систему, яка автоматично доповнює табличні дані з Excel через вебпошук (Tavily API) та LLM-витягнення.

**Архітектура:** простий pipeline без LangGraph:
1. **Читання** Excel (`pandas`)
2. **Planner** — один виклик LLM аналізує структуру таблиці й завдання → повертає план пошуку (Pydantic structured output)
3. **Researcher** — Tavily API з кешем і повторними спробами
4. **Extractor** — LLM витягує конкретне значення у JSON
5. **Validator** — приведення типів, перехресна перевірка
6. **Writer** — зберігає `*_enriched.xlsx` з логом на окремому аркуші

**Паралельність:** `ThreadPoolExecutor(max_workers=8)` + `tqdm`  
**Оціночна вартість на обох наборах:** < $0.10

## 0. Встановлення залежностей

**Мета:** підготувати середовище — `langchain-openai`, `tavily-python`, `pandas`, `openpyxl`, `tqdm`.

- `subprocess.check_call` запускає `pip` у поточному Python-середовищі.
- Параметр `-q` придушує зайві логи.
- **Не витрачає API-токени.**

In [1]:
import subprocess, sys

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "langchain-openai>=0.1.0",
    "langchain-core>=0.2.0",
    "langchain-community>=0.2.0",
    "tavily-python>=0.3.0",
    "pandas>=2.0.0",
    "openpyxl>=3.1.0",
    "tqdm>=4.66.0",
    "pydantic>=2.0.0",
])
print('✅ Всі залежності встановлено')

✅ Всі залежності встановлено


## 1. API-ключі

**Мета:** безпечно завантажити `OPENAI_API_KEY` і `TAVILY_API_KEY`.

- Спочатку шукаємо в **Colab Secrets** (`google.colab.userdata.get`).
- Якщо не знайдено — просимо ввести через `getpass` (не відображається в виводі).

In [2]:
import os
from getpass import getpass

def _get_secret(name: str) -> str:
    """Зчитати секрет: спочатку Colab Secrets, потім getpass."""
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            print(f"✅ {name} завантажено з Colab Secrets")
            return value
    except Exception:
        pass
    return getpass(f"🔑 Введіть {name}: ")

os.environ["OPENAI_API_KEY"]  = _get_secret("OPENAI_API_KEY")
os.environ["TAVILY_API_KEY"]  = _get_secret("TAVILY_API_KEY")
print("✅ Ключі встановлено")

✅ OPENAI_API_KEY завантажено з Colab Secrets
✅ TAVILY_API_KEY завантажено з Colab Secrets
✅ Ключі встановлено


## 2. Завантаження вихідних Excel-файлів за URL

**Мета:** отримати оригінальні `capitals.xlsx` та `mountains.xlsx` за посиланнями з домашнього завдання **без підключення Google Drive**.

- Код намагається завантажити обидва файли за прямими посиланнями і перевіряє, що отримано справжні файли `.xlsx`, а не HTML-сторінку авторизації.
- Якщо завантаження успішне, файли зберігаються під назвами `capitals.xlsx` і `mountains.xlsx` для наступних розділів. Наявні файли з такими назвами будуть замінені оригіналами.
- Якщо завантаження недоступне (наприклад, потрібен дозвіл на доступ), буде виведено повідомлення про помилку. **Наступний розділ створить резервні демонстраційні файли, якщо відповідних файлів немає.**

Ця комірка не використовує ключі OpenAI/Tavily і не витрачає токени LLM.

In [3]:
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile, BadZipFile
import requests

# Прямі посилання на файли з умови домашнього завдання.
excel_urls = {
    "capitals.xlsx": "https://docs.google.com/spreadsheets/d/1BGqfrrxMn4FQuY5N6Q0-zZOJ92i9ONCQ/export?format=xlsx",
    "mountains.xlsx": "https://docs.google.com/spreadsheets/d/1v8nvTqw8G-xG3sQ5JTeLiqjfoZVzof6B/export?format=xlsx"
}

downloaded = []
failed = []

for filename, url in excel_urls.items():
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        file_bytes = response.content

        # Перевірка вмісту: Google інколи повертає сторінку входу замість Excel.
        with ZipFile(BytesIO(file_bytes)) as archive:
            if "xl/workbook.xml" not in archive.namelist():
                raise ValueError("Відповідь не містить структури файлу .xlsx")

        Path(filename).write_bytes(file_bytes)
        downloaded.append(filename)
        print(f"Завантажено {filename} ({len(file_bytes):,} байтів)")
    except (requests.RequestException, BadZipFile, ValueError, OSError) as error:
        failed.append(filename)
        print(f"Не вдалося завантажити {filename}: {error}")
        print("Наступний розділ створить демонстраційний файл, якщо файл відсутній.")

print(f"Підсумок завантаження: {len(downloaded)} успішно, {len(failed)} не завантажено.")


Завантажено capitals.xlsx (9,019 байтів)
Завантажено mountains.xlsx (8,801 байтів)
Підсумок завантаження: 2 успішно, 0 не завантажено.


## 3. Резервне створення тестових Excel-файлів (якщо не завантажились)

**Мета:** створити демонстраційні `capitals.xlsx` та `mountains.xlsx` **лише якщо після попереднього розділу їх немає**.

- Якщо файл успішно завантажився за URL або вже існує, генерація цього файлу пропускається.
- Якщо завантаження не вдалося й файлу немає, створюється резервний набір із 10 прикладів для тестування алгоритму.

API-ключі та токени LLM на цьому кроці не використовуються.

In [4]:
import pandas as pd
from pathlib import Path

def _create_if_missing(path: str, df: pd.DataFrame) -> None:
    p = Path(path)
    if p.exists():
        print(f"ℹ️  {path} вже існує — пропускаємо генерацію")
        return
    df.to_excel(path, index=False)
    print(f"✅ Створено {path} ({len(df)} рядків)")

# --- capitals.xlsx ---
capitals_df = pd.DataFrame([
    {"Capital_From": "Kyiv",      "Country_From": "Ukraine",        "Capital_To": "Paris",    "Country_To": "France",   "distance": None},
    {"Capital_From": "London",    "Country_From": "United Kingdom", "Capital_To": "Rome",     "Country_To": "Italy",    "distance": None},
    {"Capital_From": "Paris",     "Country_From": "France",         "Capital_To": "Berlin",   "Country_To": "Germany",  "distance": None},
    {"Capital_From": "Berlin",    "Country_From": "Germany",        "Capital_To": "Vienna",   "Country_To": "Austria",  "distance": None},
    {"Capital_From": "Warsaw",    "Country_From": "Poland",         "Capital_To": "Kyiv",     "Country_To": "Ukraine",  "distance": None},
    {"Capital_From": "Rome",      "Country_From": "Italy",          "Capital_To": "Madrid",   "Country_To": "Spain",    "distance": None},
    {"Capital_From": "Madrid",    "Country_From": "Spain",          "Capital_To": "Lisbon",   "Country_To": "Portugal", "distance": None},
    {"Capital_From": "Vienna",    "Country_From": "Austria",        "Capital_To": "Budapest", "Country_To": "Hungary",  "distance": None},
    {"Capital_From": "Stockholm", "Country_From": "Sweden",         "Capital_To": "Oslo",     "Country_To": "Norway",   "distance": None},
    {"Capital_From": "Athens",    "Country_From": "Greece",         "Capital_To": "Istanbul", "Country_To": "Turkey",   "distance": None},
])
_create_if_missing("capitals.xlsx", capitals_df)

# --- mountains.xlsx ---
mountains_df = pd.DataFrame([
    {"Mountain": "Mount Everest",  "Country": "Nepal/China",    "height": None},
    {"Mountain": "K2",             "Country": "Pakistan/China", "height": None},
    {"Mountain": "Kangchenjunga",  "Country": "Nepal/India",    "height": None},
    {"Mountain": "Lhotse",         "Country": "Nepal/China",    "height": None},
    {"Mountain": "Makalu",         "Country": "Nepal/China",    "height": None},
    {"Mountain": "Cho Oyu",        "Country": "Nepal/China",    "height": None},
    {"Mountain": "Dhaulagiri",     "Country": "Nepal",          "height": None},
    {"Mountain": "Manaslu",        "Country": "Nepal",          "height": None},
    {"Mountain": "Nanga Parbat",   "Country": "Pakistan",       "height": None},
    {"Mountain": "Annapurna",      "Country": "Nepal",          "height": None},
])
_create_if_missing("mountains.xlsx", mountains_df)

print("\ncapitals.xlsx preview:")
display(pd.read_excel("capitals.xlsx").head(3))
print("mountains.xlsx preview:")
display(pd.read_excel("mountains.xlsx").head(3))

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


ℹ️  capitals.xlsx вже існує — пропускаємо генерацію
ℹ️  mountains.xlsx вже існує — пропускаємо генерацію

capitals.xlsx preview:


,Capital_From,Country_From,Capital_To,Country_To,distance
0,Kyiv,Ukraine,Paris,France,NaN
1,London,United Kingdom,Rome,Italy,NaN
2,Paris,France,Berlin,Germany,NaN


mountains.xlsx preview:


,Mountain,Country,height
0,Mount Everest,Nepal/China,NaN
1,K2,Pakistan/China,NaN
2,Kangchenjunga,Nepal/India,NaN


## 4. Імпорти та Pydantic-моделі

**Мета:** завантажити всі бібліотеки та визначити структури даних.

- `EnrichmentPlan` — structured output від Planner: що шукати, як шукати, якого типу дані.
- `ExtractionResult` — structured output від Extractor: значення, одиниця, впевненість, джерело.
- `RowResult` — підсумок по одному рядку для запису в лог.

In [5]:
from __future__ import annotations

import json, math, os, re, threading, time
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Literal, Optional

import pandas as pd
from pydantic import BaseModel, Field
from tqdm.notebook import tqdm

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_community.callbacks import get_openai_callback
from tavily import TavilyClient


# Pydantic-моделі

class TargetColumn(BaseModel):
    """Опис цільової колонки для заповнення."""
    name: str = Field(description="Назва колонки у таблиці")
    data_type: Literal["number", "string"] = Field(description="Тип значення")
    unit: str = Field(default="", description="Одиниця виміру (km, m …) або порожньо")


class EnrichmentPlan(BaseModel):
    """Повний план збагачення від Planner (1 виклик LLM)."""
    target_columns: List[TargetColumn]
    key_columns: List[str] = Field(
        description="Колонки, що ідентифікують рядок"
    )
    search_query_template: str = Field(
        description="Основний шаблон запиту з {Column_Name} плейсхолдерами"
    )
    alt_query_template: str = Field(
        description="Резервний шаблон запиту"
    )
    extraction_instruction: str = Field(
        description="Що саме витягнути з тексту і в яких одиницях"
    )


class ExtractionResult(BaseModel):
    """Результат витягнення одного значення."""
    value: Optional[float | str] = Field(
        default=None,
        description="Знайдене значення. None якщо не знайдено."
    )
    unit: str = Field(default="", description="Одиниця виміру")
    confidence: float = Field(default=0.0, description="Впевненість 0.0–1.0")
    source_url: str = Field(default="", description="URL джерела")
    note: str = Field(default="", description="Примітка")


@dataclass
class RowResult:
    """Підсумок обробки одного рядка."""
    row_index: int
    column: str
    query: str
    value: Any
    unit: str
    confidence: float
    source_url: str
    status: str  # found | llm_knowledge | not_found | needs_review | error
    note: str
    error: str = ""


print("✅ Імпорти та моделі визначено")

✅ Імпорти та моделі визначено


/tmp/ipykernel_1647/2903081206.py:15: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.callbacks import get_openai_callback


## 5. Planner — аналіз таблиці та побудова плану

**Мета:** один виклик LLM → повний план пошуку.

**Вхід:** список колонок, перші 3 рядки таблиці, текст задачі.  
**Вихід:** об'єкт `EnrichmentPlan` (Pydantic, через `with_structured_output`).  

- Завдяки шаблонам запитів той самий код без змін обробляє різні набори даних.
- **Витрачає: 1 виклик LLM.**

In [6]:
def build_enrichment_plan(
    df: pd.DataFrame,
    task_description: str,
    llm: ChatOpenAI,
) -> EnrichmentPlan:
    """
    Один виклик LLM → EnrichmentPlan.

    Args:
        df:               DataFrame з прочитаного Excel.
        task_description: Текстовий опис завдання від користувача.
        llm:              Ініційована LLM.

    Returns:
        EnrichmentPlan із шаблонами запитів і описом цільових колонок.
    """
    columns_info = list(df.columns)

    # Показуємо перші 3 рядки (без NaN-стовпців, щоб не збивати з пантелику)
    sample_rows = df.head(3).to_dict(orient="records")

    system_prompt = (
        "You are a data enrichment planning expert. "
        "Analyze the table structure and the user's task, then produce a concise enrichment plan. "
        "Use English for all template strings (query templates, instructions). "
        "Query templates must use {ColumnName} placeholders matching exact column names."
    )

    human_prompt = f"""
Table columns: {columns_info}

Sample rows (first 3):
{json.dumps(sample_rows, ensure_ascii=False, indent=2)}

User task: {task_description}

Based on the table and task:
1. Identify which column(s) need to be filled (they are empty/None).
2. Identify which columns uniquely identify a row (key columns) — use them in search queries.
3. Create a precise English search query template using column name placeholders in {{curly braces}}.
4. Create an alternative (backup) query template.
5. Write a clear extraction instruction for the LLM extractor.
"""

    structured_llm = llm.with_structured_output(EnrichmentPlan)
    plan: EnrichmentPlan = structured_llm.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=human_prompt),
    ])
    return plan


print("✅ Planner визначено")

✅ Planner визначено


## 6. Researcher — пошук через Tavily

**Мета:** виконати вебпошук для одного рядка, повернути сніпети.

- **Кеш** (словник): однакові запити не надсилаються двічі.
- **Retry з exponential backoff:** до 3 спроб при rate-limit або таймауті.
- `include_answer=True` повертає готову AI-відповідь Tavily — це вже короткий факт.
- Якщо Tavily недоступний — повертаємо порожній результат, Extractor спробує знання LLM.
- **Витрачає: 1 Tavily-запит на рядок (або 0 якщо кеш).**

In [7]:
_search_cache: Dict[str, dict] = {}


def search_web(
    query: str,
    tavily: TavilyClient,
    max_retries: int = 3,
) -> dict:
    """
    Виконати пошуковий запит через Tavily з кешем і повторними спробами.

    Args:
        query:       Пошуковий запит (заповнений шаблон).
        tavily:      Ініційований TavilyClient.
        max_retries: Максимальна кількість спроб.

    Returns:
        Словник з ключами: answer (str), results (list[dict]).
        При помилці повертає {'answer': '', 'results': [], 'error': str}.
    """
    cache_key = query.strip().lower()
    if cache_key in _search_cache:
        return _search_cache[cache_key]

    last_err = ""
    for attempt in range(1, max_retries + 1):
        try:
            raw = tavily.search(
                query=query,
                include_answer=True,
                max_results=5,
                search_depth="basic",
            )
            result = {
                "answer":  raw.get("answer", "") or "",
                "results": raw.get("results", []),
            }
            _search_cache[cache_key] = result
            return result
        except Exception as exc:
            last_err = str(exc)
            if attempt < max_retries:
                wait = 2 ** attempt
                time.sleep(wait)

    error_result = {"answer": "", "results": [], "error": last_err}
    return error_result


def build_snippets(search_result: dict, max_chars: int = 3000) -> str:
    """
    Зібрати текстові сніпети з результату пошуку в один рядок (≤ max_chars).

    Args:
        search_result: Словник від search_web().
        max_chars:     Максимальна довжина сумарного тексту.

    Returns:
        Рядок із відповіддю Tavily і вмістом сніпетів.
    """
    parts = []
    if search_result.get("answer"):
        parts.append(f"[Tavily answer] {search_result['answer']}")
    for r in search_result.get("results", []):
        snippet = r.get("content", "") or r.get("snippet", "")
        url = r.get("url", "")
        if snippet:
            parts.append(f"[{url}] {snippet}")
    text = "\n".join(parts)
    return text[:max_chars]


_coord_cache: Dict[str, dict] = {}
_coord_lock = threading.Lock()
_coord_last_request_ts = 0.0
_COORD_MIN_INTERVAL_SEC = 1.0


def lookup_coordinates(
    city: str,
    country: str = "",
    max_retries: int = 3,
) -> dict:
    """
    Resolve city coordinates from OpenStreetMap Nominatim.

    Args:
        city:        City name.
        country:     Country name (optional but strongly recommended).
        max_retries: Number of retry attempts for transient errors.

    Returns:
        Dict with keys:
        - coords: tuple(lat, lon) or None
        - source_url: source page URL
        - error: error message if request failed
        - query: normalized lookup query
    """
    city = str(city or "").strip()
    country = str(country or "").strip()
    if not city:
        return {"coords": None, "source_url": "", "error": "empty_city", "query": ""}

    query_text = f"{city}, {country}" if country else city
    cache_key = query_text.lower()
    if cache_key in _coord_cache:
        return _coord_cache[cache_key]

    from urllib.parse import quote_plus, urlencode
    from urllib.request import Request, urlopen

    params = {
        "q": query_text,
        "format": "jsonv2",
        "limit": 1,
    }
    url = "https://nominatim.openstreetmap.org/search?" + urlencode(params)
    source_url = "https://nominatim.openstreetmap.org/ui/search.html?q=" + quote_plus(query_text)

    global _coord_last_request_ts

    last_err = ""
    for attempt in range(1, max_retries + 1):
        try:
            # Nominatim public endpoint policy: serial requests with <=1 req/sec.
            with _coord_lock:
                if cache_key in _coord_cache:
                    return _coord_cache[cache_key]

                now = time.monotonic()
                elapsed = now - _coord_last_request_ts
                if elapsed < _COORD_MIN_INTERVAL_SEC:
                    time.sleep(_COORD_MIN_INTERVAL_SEC - elapsed)

                req = Request(
                    url,
                    headers={"User-Agent": "excel-enrichment-homework/1.0"},
                )
                _coord_last_request_ts = time.monotonic()
                with urlopen(req, timeout=12) as resp:
                    payload = json.loads(resp.read().decode("utf-8"))

                if not payload:
                    result = {
                        "coords": None,
                        "source_url": source_url,
                        "error": "",
                        "query": query_text,
                    }
                    _coord_cache[cache_key] = result
                    return result

                lat = float(payload[0]["lat"])
                lon = float(payload[0]["lon"])
                result = {
                    "coords": (lat, lon),
                    "source_url": source_url,
                    "error": "",
                    "query": query_text,
                }
                _coord_cache[cache_key] = result
                return result
        except Exception as exc:
            last_err = str(exc)
            if attempt < max_retries:
                time.sleep(2 ** attempt)

    result = {
        "coords": None,
        "source_url": source_url,
        "error": f"nominatim_error: {last_err}",
        "query": query_text,
    }
    _coord_cache[cache_key] = result
    return result


print("✅ Researcher визначено")

✅ Researcher визначено


## 7. Extractor — витягнення значення через LLM

**Мета:** з тексту сніпетів отримати конкретне значення.

**Дві функції з різними промптами** — це ключове виправлення:

- `extract_value()` — промпт: *«якщо значення немає в сніпетах — повертай null»*. Правильна поведінка для реального вебпошуку.
- `ask_llm_knowledge()` — окремий промпт: *«дай найкращу відповідь з власних знань, не кажи "не знаю"»*. Тільки цей варіант гарантує, що LLM-fallback справді повертає значення.

Раніше `ask_llm_knowledge()` просто викликав `extract_value()` з порожніми сніпетами — промпт забороняв відповідати без джерел, тому fallback майже завжди повертав `null`.  
**Витрачає: 1–3 виклики LLM на рядок.**

In [8]:
def extract_value(
    snippets: str,
    extraction_instruction: str,
    row_context: str,
    llm: ChatOpenAI,
) -> ExtractionResult:
    """
    Витягнути значення з пошукових сніпетів.

    Промпт вимагає повертати null якщо значення не знайдено в тексті —
    це правильна поведінка для реальних результатів пошуку.

    Args:
        snippets:               Текст сніпетів від Researcher.
        extraction_instruction: Що саме шукати (з EnrichmentPlan).
        row_context:            Контекст рядка (наприклад 'Kyiv → Paris').
        llm:                    Ініційована LLM.

    Returns:
        ExtractionResult.
    """
    system_prompt = (
        "You are a precise data extraction assistant. "
        "Extract the requested value ONLY from the provided search snippets. "
        "If the value is not clearly stated in the snippets, set value to null and confidence to 0. "
        "Convert units if needed (e.g. miles → km, feet → meters)."
    )
    human_prompt = f"""
Context: {row_context}
Task: {extraction_instruction}

Search snippets:
{snippets or '(no snippets available)'}

Extract the single numeric or text value. Convert to required units if necessary.
"""
    return llm.with_structured_output(ExtractionResult).invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=human_prompt),
    ])


def ask_llm_knowledge(
    extraction_instruction: str,
    row_context: str,
    llm: ChatOpenAI,
) -> ExtractionResult:
    """
    Fallback: отримати значення з власних знань LLM (окремий промпт).

    ВАЖЛИВО: цей промпт свідомо відрізняється від extract_value().
    Він явно дозволяє (і просить) LLM відповідати з тренувальних знань,
    навіть якщо відповідь не є 100% точною. Confidence 0.3–0.7 для
    приблизних знань, 0.8–1.0 для точних фактів.

    Args:
        extraction_instruction: Що шукаємо.
        row_context:            Контекст рядка.
        llm:                    Ініційована LLM.

    Returns:
        ExtractionResult з note='llm_knowledge'.
    """
    system_prompt = (
        "You are a knowledgeable assistant with broad factual knowledge. "
        "Answer using your training data — do NOT refuse or say you don't know. "
        "If you know an approximate value, provide it with confidence 0.3–0.7. "
        "If you know the exact value (well-established fact), use confidence 0.8–1.0. "
        "Only set value=null if the question is genuinely unanswerable from any knowledge source."
    )
    human_prompt = f"""
Context: {row_context}
Task: {extraction_instruction}

Use your training knowledge to provide the best answer.
Note in the 'note' field that this comes from model knowledge, not web search.
"""
    result: ExtractionResult = llm.with_structured_output(ExtractionResult).invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=human_prompt),
    ])
    # Позначаємо джерело
    result.note = "llm_knowledge" + (f": {result.note}" if result.note else "")
    return result


print("✅ Extractor визначено (окремі промпти для веб-витягнення і LLM-знань)")

✅ Extractor визначено (окремі промпти для веб-витягнення і LLM-знань)


## 8. Validator + Writer

**Мета:** привести знайдені значення до потрібного формату, перевірити їх коректність і зберегти результат у Excel.

1. **`cast_to_number`** — конвертує рядки з числами до `float`: пробіли тисяч, кома як десяткова, від'ємний знак.
2. **`validate_range`** — перевіряє реалістичність числового значення: 1–25 000 км для відстаней, 0–10 000 м для висот (точне зіставлення одиниць).
3. **Distance-path** — для прямої відстані між містами використовується не витяг числа зі сніпетів, а координати міст + обчислення haversine.
4. **`normalize_value`** — єдина точка нормалізації для не-distance колонок.
5. **`save_enriched_excel`** — зберігає аркуші `data` і `enrichment_log`.

**Статуси рядків:**
- `found` — значення знайдено і пройшло перевірки
- `llm_knowledge` — для загального pipeline вебпошук не дав значення, використано знання LLM
- `not_found` — значення не знайдено (для distance: координати не знайдені)
- `needs_review` — низька `confidence`, поза діапазоном або розбіжність > 10% між веб і LLM
- `error` — помилка API/виняток


In [9]:
# --- Unit normalization and range limits ---

UNIT_NORMALIZER: Dict[str, str] = {
    "km": "km", "kilometer": "km", "kilometers": "km",
    "kilometre": "km", "kilometres": "km",
    "m": "m", "meter": "m", "meters": "m",
    "metre": "m", "metres": "m",
    "mi": "mi", "mile": "mi", "miles": "mi",
    "ft": "ft", "foot": "ft", "feet": "ft",
}

RANGE_LIMITS: Dict[str, tuple] = {
    "km": (1, 25_000),
    "m":  (0, 10_000),
    "mi": (1, 16_000),
    "ft": (0, 30_000),
}

STATUS_ICONS: Dict[str, str] = {
    "found":         "✅",
    "llm_knowledge": "🧠",
    "not_found":     "❌",
    "needs_review":  "⚠️",
    "error":         "💥",
}


def cast_to_number(raw: Any) -> Optional[float]:
    """
    Convert raw to float handling varied decimal/thousands separators.

    Supports: spaces as thousands separator ("8 848"), comma as decimal
    ("8,5"), comma as thousands ("8,848"), combined ("1,234.5"), negatives.

    Args:
        raw: Input value of any type.

    Returns:
        float, or None if conversion fails.
    """
    if raw is None:
        return None
    if isinstance(raw, (int, float)):
        return float(raw)
    s = str(raw).strip()
    # Remove trailing unit suffix
    s = re.sub(
        r"(?i)\s*(kilometres?|kilometers?|metres?|meters?|km|mi|miles?|feet|ft)\s*$",
        "",
        s,
    )
    # Find first number token (with optional leading minus)
    m = re.search(r"-?\d[\d\s\u00a0\u202f.,]*", s)
    if not m:
        return None
    token = m.group(0)
    negative = token.startswith("-")
    # Strip all space variants
    cleaned = (
        token.lstrip("-")
        .replace(" ", "")
        .replace("\u00a0", "")
        .replace("\u202f", "")
    )
    has_comma = "," in cleaned
    has_dot = "." in cleaned
    if has_comma and has_dot:
        # Mixed separators: last one is decimal
        if cleaned.rfind(".") > cleaned.rfind(","):
            normalized = cleaned.replace(",", "")       # 1,234.5 → 1234.5
        else:
            normalized = cleaned.replace(".", "").replace(",", ".")  # 1.234,5 → 1234.5
    elif has_comma:
        parts = cleaned.split(",")
        if len(parts) >= 2 and all(len(p) == 3 for p in parts[1:]):
            normalized = cleaned.replace(",", "")       # 8,848 → 8848
        else:
            normalized = cleaned.replace(",", ".")      # 8,5 → 8.5
    elif has_dot:
        if cleaned.count(".") > 1:
            normalized = cleaned.replace(".", "")       # 1.234.567 → 1234567
        else:
            normalized = cleaned                        # 8.5 → 8.5
    else:
        normalized = cleaned
    try:
        value = float(normalized)
        return -value if negative else value
    except ValueError:
        return None


def validate_range(value: float, unit: str) -> bool:
    """
    Check whether value falls within a realistic range for the given unit.

    Uses exact unit matching via UNIT_NORMALIZER, so "m" does not
    accidentally match "km".

    Args:
        value: Numeric value.
        unit:  Unit string (e.g. "km", "meters").

    Returns:
        True if in range or unit is unrecognised.
    """
    normalized_unit = UNIT_NORMALIZER.get(unit.lower().strip())
    if normalized_unit is None:
        return True
    limits = RANGE_LIMITS.get(normalized_unit)
    if limits is None:
        return True
    lo, hi = limits
    return lo <= value <= hi


def normalize_value(raw_value: Any, data_type: str, unit: str) -> tuple:
    """
    Normalise an extracted value according to the target column type.

    For "number": parses via cast_to_number and checks validate_range.
    For "string": converts to a stripped string (no numeric logic applied).

    Args:
        raw_value: Raw value from ExtractionResult.
        data_type: "number" or "string".
        unit:      Unit string used for range check (numbers only).

    Returns:
        (final_value, in_range): final_value is None on parse failure;
                                 in_range is False only for out-of-range numbers.
    """
    if data_type == "string":
        if raw_value is None:
            return None, True
        final = str(raw_value).strip() or None
        return final, True
    # --- number branch ---
    if raw_value is None:
        return None, True
    num = cast_to_number(raw_value)
    if num is None:
        return None, True
    final = int(round(num)) if num == round(num) else round(num, 2)
    in_range = validate_range(float(final), unit)
    return final, in_range


def cross_validate(
    search_value: Optional[float],
    llm_value: Optional[float],
    tolerance: float = 0.10,
) -> bool:
    """
    Compare values from web search and LLM knowledge.

    Args:
        search_value: Value from web search.
        llm_value:    Value from LLM knowledge.
        tolerance:    Allowed relative difference (default 10%).

    Returns:
        True if both values are known and differ by more than tolerance.
    """
    if search_value is None or llm_value is None:
        return False
    if llm_value == 0:
        return False
    return abs(search_value - llm_value) / abs(llm_value) > tolerance


def save_enriched_excel(
    original_df: pd.DataFrame,
    results: List[RowResult],
    output_path: str,
) -> None:
    """
    Save enriched DataFrame to Excel with two sheets.

    Sheet "data": original structure with filled target cells.
    Sheet "enrichment_log": per-row processing details.
    Only empty target cells are filled; original values are not modified.

    Args:
        original_df:  Original DataFrame (unchanged).
        results:      List of RowResult from processing.
        output_path:  Output file path.
    """
    enriched = original_df.copy()

    for r in results:
        if r.value is not None and r.status not in ("not_found", "error"):
            current = enriched.at[r.row_index, r.column]
            if pd.isna(current) or current is None or str(current).strip() == "":
                enriched.at[r.row_index, r.column] = r.value

    log_df = pd.DataFrame([
        {
            "row_index":  r.row_index,
            "column":     r.column,
            "query":      r.query,
            "value":      r.value,
            "unit":       r.unit,
            "confidence": round(r.confidence, 3),
            "source_url": r.source_url,
            "status":     r.status,
            "note":       r.note,
            "error":      r.error,
        }
        for r in results
    ])

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        enriched.to_excel(writer, sheet_name="data", index=False)
        log_df.to_excel(writer, sheet_name="enrichment_log", index=False)

    print(f"✅ Збережено: {output_path}")


print("✅ Validator і Writer визначено")


✅ Validator і Writer визначено


## 8а. Офлайн-перевірка (без API)

**Мета:** перевірити `cast_to_number`, `validate_range` і `normalize_value` без API-запитів.

Комірка виконується локально і не впливає на вартість.

In [10]:
# --- Offline assertions: cast_to_number ---

_cases_cast = [
    ("8 848 m",    8848.0),
    ("8,5 km",     8.5),
    ("-10 m",     -10.0),
    ("8,848",      8848.0),
    ("1,234.5",    1234.5),
    ("1.234,5",    1234.5),
    (8849,         8849.0),
    (8849.0,       8849.0),
    (None,         None),
    ("no number",  None),
]
for _raw, _expected in _cases_cast:
    _got = cast_to_number(_raw)
    assert _got == _expected, f"cast_to_number({_raw!r}) = {_got}, expected {_expected}"

# --- Offline assertions: validate_range ---

assert validate_range(1000, "km")   is True,  "1000 km should be in range"
assert validate_range(99999, "km")  is False, "99999 km out of range"
assert validate_range(8849, "m")    is True,  "8849 m should be in range"
assert validate_range(-5, "m")      is False, "-5 m out of range"
assert validate_range(99999, "m")   is False, "99999 m out of range"
assert validate_range(1000, "unknown") is True, "unknown unit should pass"
# "m" must NOT match "km" (exact unit matching)
assert validate_range(99999, "km")  is False
assert validate_range(99999, "m")   is False

# --- Offline assertions: normalize_value ---

assert normalize_value(8849,    "number", "m")   == (8849,  True)
assert normalize_value("8,5",   "number", "km")  == (8.5,   True)
assert normalize_value("Paris", "string", "")    == ("Paris", True)
assert normalize_value(None,    "number", "m")   == (None,  True)
assert normalize_value(-5,      "number", "m")   == (-5,    False)
assert normalize_value("8 848 m", "number", "m") == (8848,  True)

print("✅ Усі офлайн-перевірки пройдено")


✅ Усі офлайн-перевірки пройдено


## 9. Головна функція `process_excel` + `_process_single_row`

**Мета:** об'єднати всі етапи в єдиний процес з паралельною обробкою рядків і підсумковою статистикою.

**Логіка `_process_single_row`:**

1. Якщо задача — пряма відстань між містами (`km` + пари `Capital/City From/To`), система йде по детермінованому шляху: шукає координати обох міст в інтернеті (з урахуванням країни) і рахує відстань формулою haversine в Python. Координатні запити до Nominatim виконуються послідовно (1 запит/сек) з кешуванням.
2. Для distance-задачі fallback із пам'яті LLM не використовується: якщо координати не знайдені, клітинка залишається порожньою, причина записується в `note`/`error`.
3. Для інших колонок (наприклад, `height`) залишається базовий pipeline: Tavily-пошук → Extractor → резервний запит → (за потреби) LLM fallback.
4. `normalize_value` нормалізує тип: числові колонки — через `cast_to_number`, рядкові — через `str().strip()`.
5. Статус `needs_review` ставиться для низької `confidence`, виходу за діапазон або розбіжності > 10% між вебзначенням і LLM-перевіркою.

`with get_openai_callback() as cb` перехоплює всі LLM-виклики і рахує реальні токени та вартість.


In [11]:
def _haversine_km(
    lat1: float,
    lon1: float,
    lat2: float,
    lon2: float,
) -> int:
    """Straight-line distance (km) using the haversine formula."""
    radius_km = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return round(2 * radius_km * math.asin(math.sqrt(a)))


def _pick_row_value(
    row: dict,
    must_have: List[str],
    any_of: List[str],
) -> Optional[str]:
    """Find a row value by column-name tokens."""
    for key, value in row.items():
        if value is None or str(value).strip() == "":
            continue
        key_l = key.lower()
        if all(token in key_l for token in must_have) and any(token in key_l for token in any_of):
            return str(value).strip()
    return None


def _extract_distance_route(row: dict) -> Optional[dict]:
    """Extract origin/destination city-country tuple from row columns."""
    city_from = (
        _pick_row_value(row, ["capital"], ["from", "origin"])
        or _pick_row_value(row, ["city"], ["from", "origin"])
    )
    city_to = (
        _pick_row_value(row, ["capital"], ["to", "dest"])
        or _pick_row_value(row, ["city"], ["to", "dest"])
    )

    if not city_from or not city_to:
        return None

    country_from = _pick_row_value(row, ["country"], ["from", "origin"]) or ""
    country_to = _pick_row_value(row, ["country"], ["to", "dest"]) or ""

    return {
        "city_from": city_from,
        "country_from": country_from,
        "city_to": city_to,
        "country_to": country_to,
    }


def _is_distance_coordinate_task(
    row: dict,
    target_col: TargetColumn,
    plan: EnrichmentPlan,
) -> bool:
    """Detect whether this row should use coordinate-based distance logic."""
    if target_col.data_type != "number":
        return False

    unit = UNIT_NORMALIZER.get((target_col.unit or "").lower().strip(), target_col.unit.lower().strip())
    if unit != "km":
        return False

    route = _extract_distance_route(row)
    if route is None:
        return False

    hints_text = (
        f"{target_col.name} {plan.extraction_instruction} "
        f"{plan.search_query_template} {plan.alt_query_template}"
    ).lower()
    distance_hints = ["distance", "відстан", "straight line", "great-circle"]
    return any(h in hints_text for h in distance_hints)


def _process_single_row(
    row_index: int,
    row: dict,
    target_col: TargetColumn,
    plan: EnrichmentPlan,
    tavily: TavilyClient,
    llm: ChatOpenAI,
) -> RowResult:
    """
    Process one DataFrame row.

    For distance tasks in km with city-pair columns, coordinates are resolved
    online (city + country) and distance is computed in Python via haversine.
    If coordinates are missing, value remains empty with explicit reason.
    """
    search_errors: List[str] = []
    query = ""

    try:
        row_context = " | ".join(
            f"{k}: {v}" for k, v in row.items() if v is not None and str(v).strip()
        )

        # --- Special deterministic path for city-to-city distance ---
        if _is_distance_coordinate_task(row, target_col, plan):
            route = _extract_distance_route(row) or {}
            city_from = route.get("city_from", "")
            country_from = route.get("country_from", "")
            city_to = route.get("city_to", "")
            country_to = route.get("country_to", "")

            query = (
                f"coordinates lookup: {city_from}, {country_from} -> "
                f"{city_to}, {country_to}"
            )

            c_from = lookup_coordinates(city_from, country_from)
            c_to = lookup_coordinates(city_to, country_to)

            source_parts = [p for p in [c_from.get("source_url", ""), c_to.get("source_url", "")] if p]
            source_url = " | ".join(source_parts)

            if c_from.get("error") or c_to.get("error"):
                error_parts = [p for p in [c_from.get("error", ""), c_to.get("error", "")] if p]
                return RowResult(
                    row_index=row_index,
                    column=target_col.name,
                    query=query,
                    value=None,
                    unit="km",
                    confidence=0.0,
                    source_url=source_url,
                    status="error",
                    note="coordinate_lookup_failed",
                    error="; ".join(error_parts),
                )

            missing = []
            if c_from.get("coords") is None:
                missing.append(f"coordinates_not_found: {city_from}, {country_from}".strip(", "))
            if c_to.get("coords") is None:
                missing.append(f"coordinates_not_found: {city_to}, {country_to}".strip(", "))
            if missing:
                return RowResult(
                    row_index=row_index,
                    column=target_col.name,
                    query=query,
                    value=None,
                    unit="km",
                    confidence=0.0,
                    source_url=source_url,
                    status="not_found",
                    note="; ".join(missing),
                    error="",
                )

            lat1, lon1 = c_from["coords"]
            lat2, lon2 = c_to["coords"]
            distance_km = _haversine_km(lat1, lon1, lat2, lon2)
            in_range = validate_range(float(distance_km), "km")
            status = "found" if in_range else "needs_review"

            return RowResult(
                row_index=row_index,
                column=target_col.name,
                query=query,
                value=distance_km,
                unit="km",
                confidence=1.0,
                source_url=source_url,
                status=status,
                note="computed_from_coordinates_haversine",
                error="",
            )

        # --- 1. Build query ---
        try:
            query = plan.search_query_template.format(**row)
        except KeyError:
            key_vals = " ".join(str(row.get(k, "")) for k in plan.key_columns)
            query = f"{key_vals} {target_col.unit}"

        # --- 2. Primary web search ---
        search_result = search_web(query, tavily)
        if "error" in search_result:
            search_errors.append(search_result["error"])
        ext = extract_value(
            build_snippets(search_result),
            plan.extraction_instruction,
            row_context,
            llm,
        )

        # --- 3. Fallback query if low confidence ---
        if ext.value is None or ext.confidence < 0.3:
            try:
                alt_query = plan.alt_query_template.format(**row)
            except KeyError:
                key_vals = " ".join(str(row.get(k, "")) for k in plan.key_columns)
                alt_query = f"{key_vals} {target_col.name}"
            alt_result = search_web(alt_query, tavily)
            if "error" in alt_result:
                search_errors.append(alt_result["error"])
            alt_ext = extract_value(
                build_snippets(alt_result),
                plan.extraction_instruction,
                row_context,
                llm,
            )
            if alt_ext.value is not None and alt_ext.confidence > (ext.confidence or 0):
                ext = alt_ext
                query = alt_query

        # --- 4. LLM knowledge (informational cross-check; isolated) ---
        llm_ext: Optional[ExtractionResult] = None
        llm_num: Optional[float] = None
        llm_check_note = ""
        try:
            llm_ext = ask_llm_knowledge(plan.extraction_instruction, row_context, llm)
            if target_col.data_type == "number" and llm_ext.value is not None:
                llm_num = cast_to_number(llm_ext.value)
        except Exception as llm_exc:
            llm_check_note = f"llm_check_failed: {llm_exc}"

        # --- 5. If web search gave nothing → use LLM knowledge as primary ---
        used_llm_fallback = False
        if (ext.value is None or ext.confidence < 0.2) and llm_ext is not None:
            ext = llm_ext
            used_llm_fallback = True

        # --- 6. Normalise value (type-safe; string columns skip numeric logic) ---
        final_value, in_range = normalize_value(
            ext.value,
            target_col.data_type,
            ext.unit or target_col.unit,
        )
        search_num: Optional[float] = (
            cast_to_number(final_value) if target_col.data_type == "number" else None
        )

        # --- 7. Determine status ---
        cross_mismatch = (
            target_col.data_type == "number"
            and search_num is not None
            and llm_num is not None
            and cross_validate(search_num, llm_num)
        )

        err_text = ""
        if final_value is None:
            if search_errors:
                status = "error"
                err_text = "; ".join(search_errors)
            else:
                status = "not_found"
        elif used_llm_fallback:
            status = "llm_knowledge"
        elif (
            ext.confidence < 0.5
            or (target_col.data_type == "number" and not in_range)
            or cross_mismatch
        ):
            status = "needs_review"
        else:
            status = "found"

        # --- 8. Build note ---
        note = ext.note or ""
        if llm_check_note:
            note = f"{note}; {llm_check_note}".lstrip("; ")
        elif cross_mismatch:
            diff_pct = abs(search_num - llm_num) / abs(llm_num) * 100
            note = f"{note}; llm_check={llm_num}, diff={diff_pct:.1f}%".lstrip("; ")

        partial_err = "; ".join(search_errors) if search_errors and final_value is not None else ""

        return RowResult(
            row_index=row_index,
            column=target_col.name,
            query=query,
            value=final_value,
            unit=ext.unit,
            confidence=ext.confidence,
            source_url=ext.source_url,
            status=status,
            note=note,
            error=err_text or partial_err,
        )

    except Exception as exc:
        return RowResult(
            row_index=row_index,
            column=target_col.name,
            query=query,
            value=None,
            unit="",
            confidence=0.0,
            source_url="",
            status="error",
            note="",
            error=str(exc),
        )


def process_excel(
    file_path: str,
    task_description: str,
    output_path: Optional[str] = None,
    max_workers: int = 8,
    model: str = "gpt-4o-mini",
    checkpoint_every: int = 50,
) -> str:
    """
    Enrich an Excel file via web search and LLM.

    Reads the file, plans enrichment (Planner), processes each empty cell
    in parallel, and saves the result.
    """
    if not Path(file_path).exists():
        raise FileNotFoundError(f"❌ Файл не знайдено: {file_path}")
    if not os.environ.get("OPENAI_API_KEY"):
        raise RuntimeError("❌ OPENAI_API_KEY не встановлено")
    if not os.environ.get("TAVILY_API_KEY"):
        raise RuntimeError("❌ TAVILY_API_KEY не встановлено")

    if output_path is None:
        p = Path(file_path)
        output_path = str(p.parent / f"{p.stem}_enriched{p.suffix}")

    print(f"📂 Читання: {file_path}")
    df = pd.read_excel(file_path)
    print(f"   Рядків: {len(df)}, Колонки: {list(df.columns)}")

    llm = ChatOpenAI(model=model, temperature=0)
    tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

    with get_openai_callback() as cb:
        print("🧠 Planner: аналіз таблиці...")
        plan = build_enrichment_plan(df, task_description, llm)
        print(f"   Цільові колонки : {[c.name for c in plan.target_columns]}")
        print(f"   Ключові колонки : {plan.key_columns}")
        print(f"   Шаблон запиту   : {plan.search_query_template}")
        print(f"   Резервний       : {plan.alt_query_template}")

        if not plan.target_columns:
            raise ValueError(
                "❌ Planner не визначив цільові колонки. "
                "Перевірте task_description і структуру файлу."
            )

        rows_to_process: List[tuple] = []
        for target_col in plan.target_columns:
            if target_col.name not in df.columns:
                print(f"⚠️  Колонка '{target_col.name}' відсутня — буде додана")
                df[target_col.name] = None
            for idx, row in df.iterrows():
                val = row.get(target_col.name)
                if pd.isna(val) or val is None or str(val).strip() == "":
                    rows_to_process.append((idx, row.to_dict(), target_col))

        total = len(rows_to_process)
        print(f"\n🔍 Рядків для збагачення: {total}")

        if total == 0:
            print("ℹ️  Усі цільові клітинки вже заповнені")
            df.to_excel(output_path, index=False)
            return output_path

        _search_cache.clear()
        _coord_cache.clear()
        all_results: List[RowResult] = []
        completed = 0

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {
                executor.submit(
                    _process_single_row,
                    idx, row, target_col, plan, tavily, llm,
                ): (idx, target_col.name)
                for idx, row, target_col in rows_to_process
            }

            with tqdm(total=total, desc="Збагачення", unit="row") as pbar:
                for future in as_completed(futures):
                    result = future.result()
                    all_results.append(result)
                    completed += 1
                    pbar.update(1)

                    if checkpoint_every > 0 and completed % checkpoint_every == 0:
                        save_enriched_excel(df, all_results, output_path)
                        print(f"   💾 Checkpoint: {completed}/{total}")

    save_enriched_excel(df, all_results, output_path)

    status_counts: Dict[str, int] = {}
    for r in all_results:
        status_counts[r.status] = status_counts.get(r.status, 0) + 1

    print("\n📊 Підсумок:")
    for status, count in sorted(status_counts.items()):
        print(f"   {STATUS_ICONS.get(status, '❓')} {status}: {count}")

    print(f"   🔁 Унікальних Tavily-запитів: {len(_search_cache)}")

    print(f"\n💰 Реальна вартість LLM (get_openai_callback):")
    print(f"   Всього токенів:   {cb.total_tokens:,}")
    print(f"   Вхідних токенів:  {cb.prompt_tokens:,}")
    print(f"   Вихідних токенів: {cb.completion_tokens:,}")
    print(f"   Загальна вартість: ${cb.total_cost:.4f}")
    print(f"\n✅ Результат: {output_path}")

    return output_path


print("✅ process_excel визначено")


✅ process_excel визначено


## 10. Запуск: capitals.xlsx

**Завдання:** знайти пряму відстань між столицями в км для колонки `distance`.

**Очікуваний результат:** файл `capitals_enriched_chirkova.xlsx` зі заповненою колонкою `distance`.

In [12]:
capitals_output = process_excel(
    file_path="capitals.xlsx",
    task_description="знайди пряму відстань між столицями в км для колонки distance",
    output_path="capitals_enriched_chirkova.xlsx",
)

# Перегляд результату
result_df = pd.read_excel(capitals_output, sheet_name="data")
print("\nРезультат (capitals_enriched_chirkova.xlsx):")
display(result_df)

📂 Читання: capitals.xlsx
   Рядків: 10, Колонки: ['Capital_From', 'Country_From', 'Capital_To', 'Country_To', 'distance']
🧠 Planner: аналіз таблиці...
   Цільові колонки : ['distance']
   Ключові колонки : ['Capital_From', 'Country_From', 'Capital_To', 'Country_To']
   Шаблон запиту   : Calculate the direct distance in kilometers between {Capital_From}, {Country_From} and {Capital_To}, {Country_To}.
   Резервний       : Find the distance in kilometers from {Capital_From} in {Country_From} to {Capital_To} in {Country_To}.

🔍 Рядків для збагачення: 10


Збагачення:   0%|          | 0/10 [00:00<?, ?row/s]

✅ Збережено: capitals_enriched_chirkova.xlsx

📊 Підсумок:
   ✅ found: 10
   🔁 Унікальних Tavily-запитів: 0

💰 Реальна вартість LLM (get_openai_callback):
   Всього токенів:   1,413
   Вхідних токенів:  1,304
   Вихідних токенів: 109
   Загальна вартість: $0.0003

✅ Результат: capitals_enriched_chirkova.xlsx

Результат (capitals_enriched_chirkova.xlsx):


,Capital_From,Country_From,Capital_To,Country_To,distance
0,Kyiv,Ukraine,Paris,France,2024
1,London,United Kingdom,Rome,Italy,1434
2,Paris,France,Berlin,Germany,877
3,Berlin,Germany,Vienna,Austria,524
4,Warsaw,Poland,Kyiv,Ukraine,690
5,Rome,Italy,Madrid,Spain,1363
6,Madrid,Spain,Lisbon,Portugal,503
7,Vienna,Austria,Budapest,Hungary,214
8,Stockholm,Sweden,Oslo,Norway,417
9,Athens,Greece,Istanbul,Turkey,562


## 11. Запуск: mountains.xlsx

**Завдання:** додати висоту гір у метрах до колонки `height`.

**Очікуваний результат:** файл `mountains_enriched_chirkova.xlsx` зі заповненою колонкою `height`.

In [13]:
mountains_output = process_excel(
    file_path="mountains.xlsx",
    task_description="додай висоту гір у метрах до колонки height",
    output_path="mountains_enriched_chirkova.xlsx",
)

# Перегляд результату
result_df = pd.read_excel(mountains_output, sheet_name="data")
print("\nРезультат (mountains_enriched_chirkova.xlsx):")
display(result_df)

📂 Читання: mountains.xlsx
   Рядків: 10, Колонки: ['Mountain', 'Country', 'height']
🧠 Planner: аналіз таблиці...
   Цільові колонки : ['height']
   Ключові колонки : ['Mountain', 'Country']
   Шаблон запиту   : What is the height of the mountain {Mountain} located in {Country}?
   Резервний       : Provide the height in meters for {Mountain} in {Country}.

🔍 Рядків для збагачення: 10


Збагачення:   0%|          | 0/10 [00:00<?, ?row/s]

✅ Збережено: mountains_enriched_chirkova.xlsx

📊 Підсумок:
   ✅ found: 10
   🔁 Унікальних Tavily-запитів: 10

💰 Реальна вартість LLM (get_openai_callback):
   Всього токенів:   1,329
   Вхідних токенів:  1,247
   Вихідних токенів: 82
   Загальна вартість: $0.0002

✅ Результат: mountains_enriched_chirkova.xlsx

Результат (mountains_enriched_chirkova.xlsx):


,Mountain,Country,height
0,Mount Everest,Nepal/China,8848.86
1,K2,Pakistan/China,8611.00
2,Kangchenjunga,Nepal/India,8586.00
3,Lhotse,Nepal/China,8516.00
4,Makalu,Nepal/China,8485.00
5,Cho Oyu,Nepal/China,8188.00
6,Dhaulagiri,Nepal,8167.00
7,Manaslu,Nepal,8163.00
8,Nanga Parbat,Pakistan,8126.00
9,Annapurna,Nepal,8091.00


## 12. Перевірка точності

**Мета:** автоматично порівняти знайдені значення з еталонними без додаткових API-запитів.

- Відстані між столицями перераховуємо формулою **haversine** за відомими координатами.
- Висоти гір беремо з перевіреного довідника (UIAA / Wikipedia).
- Шкала оцінювання відповідно до умови завдання: **40 балів лише якщо всі значення в межах ±10%**.

In [14]:
# --- Haversine formula ---

def haversine(
    lat1: float,
    lon1: float,
    lat2: float,
    lon2: float,
) -> float:
    """
    Straight-line distance between two surface points in km.

    Args:
        lat1, lon1: Coordinates of point A (degrees).
        lat2, lon2: Coordinates of point B (degrees).

    Returns:
        Distance in km (rounded to nearest integer).
    """
    R = 6371.0
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi    = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2
    return round(2 * R * math.asin(math.sqrt(a)))


# --- Reference data ---

CAPITAL_COORDS = {
    "Kyiv":      (50.450, 30.523),
    "Paris":     (48.857,  2.352),
    "London":    (51.508, -0.128),
    "Rome":      (41.902, 12.496),
    "Berlin":    (52.520, 13.405),
    "Vienna":    (48.208, 16.373),
    "Warsaw":    (52.237, 21.017),
    "Madrid":    (40.417, -3.704),
    "Lisbon":    (38.717, -9.139),
    "Budapest":  (47.498, 19.040),
    "Stockholm": (59.333, 18.065),
    "Oslo":      (59.913, 10.752),
    "Athens":    (37.979, 23.716),
    "Istanbul":  (41.015, 28.979),
}

MOUNTAIN_HEIGHTS_REFERENCE = {
    "Mount Everest":  8849,
    "K2":             8611,
    "Kangchenjunga":  8586,
    "Lhotse":         8516,
    "Makalu":         8485,
    "Cho Oyu":        8188,
    "Dhaulagiri":     8167,
    "Manaslu":        8163,
    "Nanga Parbat":   8126,
    "Annapurna":      8091,
}


# --- Accuracy check ---

def check_accuracy(
    result_path: str,
    reference: dict,
    key_col: str,
    value_col: str,
    tolerance: float = 0.10,
) -> None:
    """
    Compare enriched values against known reference values.

    Args:
        result_path: Path to the enriched xlsx file.
        reference:   Dict mapping key → expected value.
        key_col:     Name of the key column in the DataFrame.
        value_col:   Name of the column containing enriched values.
        tolerance:   Allowed relative deviation (default 10%).
    """
    df = pd.read_excel(result_path, sheet_name="data")
    total = 0
    ok = 0
    rows = []

    for _, row in df.iterrows():
        key = str(row[key_col]).strip()
        ref = reference.get(key)
        if ref is None:
            continue
        total += 1
        got = row[value_col]
        if pd.isna(got) or got is None:
            rows.append({key_col: key, "got": "—", "reference": ref,
                         "within_10%": "❌", "diff%": "N/A"})
            continue
        got_num = float(got)
        within = abs(got_num - ref) / ref <= tolerance
        if within:
            ok += 1
        diff_pct = f"{abs(got_num - ref) / ref * 100:.1f}%"
        rows.append({
            key_col:     key,
            "got":       int(got_num),
            "reference": ref,
            "within_10%": "✅" if within else "❌",
            "diff%":     diff_pct,
        })

    display(pd.DataFrame(rows))
    pct = ok / total * 100 if total else 0
    print(f"\nТочність: {ok}/{total} = {pct:.1f}% в межах ±{int(tolerance * 100)}%")
    if pct == 100.0:
        print("🏆 Відмінно (100%) — 40 балів")
    elif pct >= 80:
        print("✅ Добре (80–99%) — 30 балів")
    elif pct >= 60:
        print("⚠️  Задовільно (60–79%) — 20 балів")
    else:
        print("❌ Незадовільно (<60%) — 10 балів")


# --- Run checks ---

print("=" * 60)
print("ТОЧНІСТЬ — capitals_enriched_chirkova.xlsx")
print("=" * 60)

capitals_df_check = pd.read_excel("capitals.xlsx")
caps_reference = {}
for _, row in capitals_df_check.iterrows():
    c1 = str(row["Capital_From"]).strip()
    c2 = str(row["Capital_To"]).strip()
    if c1 in CAPITAL_COORDS and c2 in CAPITAL_COORDS:
        caps_reference[c1] = haversine(*CAPITAL_COORDS[c1], *CAPITAL_COORDS[c2])

check_accuracy(
    result_path="capitals_enriched_chirkova.xlsx",
    reference=caps_reference,
    key_col="Capital_From",
    value_col="distance",
)

print()
print("=" * 60)
print("ТОЧНІСТЬ — mountains_enriched_chirkova.xlsx")
print("=" * 60)

check_accuracy(
    result_path="mountains_enriched_chirkova.xlsx",
    reference=MOUNTAIN_HEIGHTS_REFERENCE,
    key_col="Mountain",
    value_col="height",
)


ТОЧНІСТЬ — capitals_enriched_chirkova.xlsx


,Capital_From,got,reference,within_10%,diff%
0,Kyiv,2024,2023,✅,0.0%
1,London,1434,1434,✅,0.0%
2,Paris,877,877,✅,0.0%
3,Berlin,524,524,✅,0.0%
4,Warsaw,690,689,✅,0.1%
5,Rome,1363,1364,✅,0.1%
6,Madrid,503,503,✅,0.0%
7,Vienna,214,214,✅,0.0%
8,Stockholm,417,416,✅,0.2%
9,Athens,562,564,✅,0.4%



Точність: 10/10 = 100.0% в межах ±10%
🏆 Відмінно (100%) — 40 балів

ТОЧНІСТЬ — mountains_enriched_chirkova.xlsx


,Mountain,got,reference,within_10%,diff%
0,Mount Everest,8848,8849,✅,0.0%
1,K2,8611,8611,✅,0.0%
2,Kangchenjunga,8586,8586,✅,0.0%
3,Lhotse,8516,8516,✅,0.0%
4,Makalu,8485,8485,✅,0.0%
5,Cho Oyu,8188,8188,✅,0.0%
6,Dhaulagiri,8167,8167,✅,0.0%
7,Manaslu,8163,8163,✅,0.0%
8,Nanga Parbat,8126,8126,✅,0.0%
9,Annapurna,8091,8091,✅,0.0%



Точність: 10/10 = 100.0% в межах ±10%
🏆 Відмінно (100%) — 40 балів


## 13. Фінальний огляд і завантаження результатів

**Мета:** переглянути збагачені таблиці, перевірити рядки зі статусом `needs_review` та завантажити файли.

**Що виконується:**
1. Перевірка існування вихідних файлів.
2. Відображення фінальних таблиць (аркуш `data`).
3. Аналіз логу збагачення: статистика по статусах, виділення `needs_review`-рядків для ручної перевірки.
4. Завантаження файлів через `files.download` (Colab).

> `needs_review` означає, що значення від Tavily і значення з власних знань LLM розійшлися більш ніж на 10%, або значення виходить за межі реалістичного діапазону. Такі рядки рекомендується перевірити вручну.

In [15]:
from google.colab import files

OUTPUT_FILES = [
    ("capitals_enriched_chirkova.xlsx",  "Capital_From", "distance"),
    ("mountains_enriched_chirkova.xlsx", "Mountain",     "height"),
]

for filepath, key_col, value_col in OUTPUT_FILES:
    p = Path(filepath)
    print("=" * 60)

    # --- Check file exists ---
    if not p.exists():
        print(f"❌ Файл не знайдено: {filepath}")
        print("   Переконайтесь, що секції 10 і 11 виконані.")
        continue
    print(f"✅ {filepath}  ({p.stat().st_size / 1024:.1f} KB)")

    # --- Final enriched table ---
    data_df = pd.read_excel(filepath, sheet_name="data")
    print(f"\nАркуш 'data' — {len(data_df)} рядків:")
    display(data_df)

    # --- Enrichment log summary ---
    log_df = pd.read_excel(filepath, sheet_name="enrichment_log")
    status_counts = log_df["status"].value_counts()
    print("\nСтатистика збагачення:")
    for status, count in status_counts.items():
        print(f"   {STATUS_ICONS.get(status, '❓')} {status}: {count}")

    # --- Rows requiring manual review ---
    review_df = log_df[log_df["status"] == "needs_review"]
    if not review_df.empty:
        print(f"\n⚠️  {len(review_df)} рядок(ів) потребують ручної перевірки (needs_review):")
        display(review_df[["row_index", "query", "value", "confidence", "note"]])
    else:
        print("\n✅ Рядків з needs_review немає")

    # --- Rows with errors ---
    error_df = log_df[log_df["status"] == "error"]
    if not error_df.empty:
        print(f"\n💥 {len(error_df)} рядок(ів) завершились помилкою:")
        display(error_df[["row_index", "error"]])

print("=" * 60)
print()

# --- Download files ---
print("📥 Завантаження файлів...")
for filepath, _, __ in OUTPUT_FILES:
    if Path(filepath).exists():
        files.download(filepath)
        print(f"   ✅ {filepath}")
    else:
        print(f"   ❌ {filepath} — файл відсутній, пропускаємо")


✅ capitals_enriched_chirkova.xlsx  (6.6 KB)

Аркуш 'data' — 10 рядків:


,Capital_From,Country_From,Capital_To,Country_To,distance
0,Kyiv,Ukraine,Paris,France,2024
1,London,United Kingdom,Rome,Italy,1434
2,Paris,France,Berlin,Germany,877
3,Berlin,Germany,Vienna,Austria,524
4,Warsaw,Poland,Kyiv,Ukraine,690
5,Rome,Italy,Madrid,Spain,1363
6,Madrid,Spain,Lisbon,Portugal,503
7,Vienna,Austria,Budapest,Hungary,214
8,Stockholm,Sweden,Oslo,Norway,417
9,Athens,Greece,Istanbul,Turkey,562



Статистика збагачення:
   ✅ found: 10

✅ Рядків з needs_review немає
✅ mountains_enriched_chirkova.xlsx  (6.6 KB)

Аркуш 'data' — 10 рядків:


,Mountain,Country,height
0,Mount Everest,Nepal/China,8848.86
1,K2,Pakistan/China,8611.00
2,Kangchenjunga,Nepal/India,8586.00
3,Lhotse,Nepal/China,8516.00
4,Makalu,Nepal/China,8485.00
5,Cho Oyu,Nepal/China,8188.00
6,Dhaulagiri,Nepal,8167.00
7,Manaslu,Nepal,8163.00
8,Nanga Parbat,Pakistan,8126.00
9,Annapurna,Nepal,8091.00



Статистика збагачення:
   ✅ found: 10

✅ Рядків з needs_review немає

📥 Завантаження файлів...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

   ✅ capitals_enriched_chirkova.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

   ✅ mountains_enriched_chirkova.xlsx


## 14. Перевірка на 1000 реальних записів

Тестові файли: `capitals_1000_unique.xlsx`, `mountains_1000_unique.xlsx`, `references_real_1000.xlsx`.

- Столиці: 1000 різних пар із 54 реальних столиць, без зворотних дублікатів.
- Гори й вершини: 1000 різних записів GeoNames із п'яти країн; висоту взято з поля elevation, не з DEM.
- Еталони зберігаються окремо й не передаються моделі. Відстані в еталонах обчислено за координатами GeoNames; джерело координат у системі — Nominatim, тому допускається ±10%.
- Назви, країни, GeoNames ID і посилання збережені у файлі еталонів. Дані не є незалежно перевіреним геодезичним вимірюванням.

Перевіряються кількість і порядок рядків, незмінність початкових даних, повнота журналу, заповнення, точність ±10%, помилки та тривалість. Тест не змінює звичайні тестові файли.

In [17]:
# Налаштування: спочатку перевіримо столиці, потім окремо гори. Столиці обробити швидше.
RUN_1000_TEST = True  # Зміни на True для запуску.
TEST_DATASETS = ["mountains"]  # Або ["mountains"], або ["capitals", "mountains"].

# Посилання на файли
TEST_INPUT_URLS = {"capitals": "https://docs.google.com/spreadsheets/d/1crtFzqQ-KBOA9c4K0iEWl9rWsTh3aCgA/edit?usp=sharing&ouid=110271482628184775666&rtpof=true&sd=true",
                   "mountains": "https://docs.google.com/spreadsheets/d/11wOd2UTkm4fsXdMLgIoShkqWmFQwliLr/edit?usp=sharing&ouid=110271482628184775666&rtpof=true&sd=true"}
TEST_REFERENCE_URL = "https://docs.google.com/spreadsheets/d/1lha1purMrtLvWB0KF8hsT2MCaJG_4lI8/edit?usp=sharing&ouid=110271482628184775666&rtpof=true&sd=true"

import io as _test_io
import re as _test_re
import zipfile as _test_zipfile
from pathlib import Path as _TestPath
from urllib.request import Request as _TestRequest, urlopen as _test_urlopen
from time import perf_counter as _test_perf_counter
import pandas as _test_pd

def _test_download_if_needed(filename, url):
    path = _TestPath(filename)
    if path.exists():
        return
    if not url.strip():
        raise FileNotFoundError(
            f"Завантаж {filename} у поточну папку Colab або додай URL у налаштуваннях."
        )
    sheet = _test_re.search(r"docs\.google\.com/spreadsheets/d/([^/?#]+)", url)
    drive = _test_re.search(r"drive\.google\.com/file/d/([^/?#]+)", url)
    if sheet:
        url = f"https://docs.google.com/spreadsheets/d/{sheet.group(1)}/export?format=xlsx"
    elif drive:
        url = f"https://drive.google.com/uc?export=download&id={drive.group(1)}"
    with _test_urlopen(_TestRequest(url, headers={"User-Agent": "excel-enrichment-test/1.0"}), timeout=60) as response:
        data = response.read()
    try:
        with _test_zipfile.ZipFile(_test_io.BytesIO(data)) as archive:
            if "xl/workbook.xml" not in archive.namelist():
                raise ValueError("Немає структури Excel")
    except (_test_zipfile.BadZipFile, ValueError) as exc:
        raise ValueError(
            f"URL для {filename} не повернув XLSX. Перевір доступ за посиланням."
        ) from exc
    path.write_bytes(data)

def _test_validate_1000(before, after, log, reference, keys, target, expected_column):
    assert len(before) == 1000, f"У вхідному файлі {len(before)} рядків замість 1000"
    assert len(after) == 1000, f"У результаті {len(after)} рядків замість 1000"
    assert len(log) == 1000, f"У журналі {len(log)} рядків замість 1000"
    assert len(reference) == 1000, "Неправильний файл еталонів"
    assert before[target].isna().all(), "Цільова колонка має бути порожньою перед тестом"
    assert not before[keys].isna().any().any(), "Порожній ідентифікатор у вхідних даних"
    assert not before.duplicated(keys).any(), "У тестових даних є дублікати"
    assert list(after.columns) == list(before.columns), "Змінилися колонки або їх порядок"
    _test_pd.testing.assert_frame_equal(before[keys], after[keys], check_dtype=False)
    assert not log[["row_index", "column"]].duplicated().any(), "Дублікати в журналі"
    assert set(log["row_index"]) == set(range(1000)), "Неповне покриття індексів у журналі"
    assert log["column"].eq(target).all(), "Неправильна цільова колонка в журналі"
    joined = after.merge(reference[keys + [expected_column]], on=keys, how="left", validate="one_to_one")
    assert joined[expected_column].notna().all(), "Для частини рядків немає еталона"
    actual = _test_pd.to_numeric(joined[target], errors="coerce")
    expected = _test_pd.to_numeric(joined[expected_column], errors="raise")
    relative_error = (actual - expected).abs() / expected.abs()
    within = actual.notna() & relative_error.le(0.10)
    summary = {
        "rows": len(after),
        "numeric_values": int(actual.notna().sum()),
        "within_10_percent": int(within.sum()),
        "accuracy_percent": round(float(within.mean() * 100), 1),
        "api_error_rows": int(log["status"].eq("error").sum()),
        "not_found_rows": int(log["status"].eq("not_found").sum()),
        "needs_review_rows": int(log["status"].eq("needs_review").sum()),
        "llm_knowledge_rows": int(log["status"].eq("llm_knowledge").sum()),
        "original_values_preserved": True,
    }
    failures = joined.loc[~within, keys + [target, expected_column]].copy()
    failures["deviation_percent"] = (relative_error[~within] * 100).round(1)
    return summary, failures

if RUN_1000_TEST:
    assert "process_excel" in globals(), "Спочатку виконай комірки з функціями системи"
    assert TEST_DATASETS and set(TEST_DATASETS).issubset({"capitals", "mountains"})
    _test_download_if_needed("references_real_1000.xlsx", TEST_REFERENCE_URL)
    _test_summaries = []
    for _test_kind in TEST_DATASETS:
        _test_is_capitals = _test_kind == "capitals"
        _test_input = f"{_test_kind}_1000_unique.xlsx"
        _test_target = "distance" if _test_is_capitals else "height"
        _test_keys = ["Capital_From", "Country_From", "Capital_To", "Country_To"] if _test_is_capitals else ["Mountain", "Country"]
        _test_expected = "expected_distance_km" if _test_is_capitals else "expected_height_m"
        _test_task = "знайди пряму відстань між столицями в км для колонки distance" if _test_is_capitals else "додай висоту гір у метрах до колонки height"
        _test_download_if_needed(_test_input, TEST_INPUT_URLS[_test_kind])
        _test_before = _test_pd.read_excel(_test_input, sheet_name="data")
        _test_reference = _test_pd.read_excel("references_real_1000.xlsx", sheet_name=f"{_test_kind}_reference")
        _test_start = _test_perf_counter()
        _test_output = process_excel(file_path=_test_input, task_description=_test_task, output_path=f"{_test_kind}_1000_unique_enriched.xlsx")
        _test_seconds = _test_perf_counter() - _test_start
        _test_after = _test_pd.read_excel(_test_output, sheet_name="data")
        _test_log = _test_pd.read_excel(_test_output, sheet_name="enrichment_log")
        _test_summary, _test_failures = _test_validate_1000(_test_before, _test_after, _test_log, _test_reference, _test_keys, _test_target, _test_expected)
        _test_summary.update(dataset=_test_kind, seconds=round(_test_seconds, 1))
        _test_summaries.append(_test_summary)
        print(f"\n{_test_kind}: структура й початкові дані збережені; {_test_summary['within_10_percent']}/1000 значень у межах ±10%; {_test_seconds:.1f} с")
        if not _test_failures.empty:
            print("Перші 20 значень, які не пройшли перевірку:")
            display(_test_failures.head(20))
    display(_test_pd.DataFrame(_test_summaries))
else:
    print("Тест вимкнено. Для запуску встанови RUN_1000_TEST = True. Почни з TEST_DATASETS = ['capitals'].")


📂 Читання: mountains_1000_unique.xlsx
   Рядків: 1000, Колонки: ['Mountain', 'Country', 'height']
🧠 Planner: аналіз таблиці...
   Цільові колонки : ['height']
   Ключові колонки : ['Mountain', 'Country']
   Шаблон запиту   : What is the height of the mountain {Mountain} located in {Country} in meters?
   Резервний       : Please provide the elevation of {Mountain} in {Country} in meters.

🔍 Рядків для збагачення: 1000


Збагачення:   0%|          | 0/1000 [00:00<?, ?row/s]

✅ Збережено: mountains_1000_unique_enriched.xlsx
   💾 Checkpoint: 50/1000
✅ Збережено: mountains_1000_unique_enriched.xlsx
   💾 Checkpoint: 100/1000
✅ Збережено: mountains_1000_unique_enriched.xlsx
   💾 Checkpoint: 150/1000
✅ Збережено: mountains_1000_unique_enriched.xlsx
   💾 Checkpoint: 200/1000
✅ Збережено: mountains_1000_unique_enriched.xlsx
   💾 Checkpoint: 250/1000
✅ Збережено: mountains_1000_unique_enriched.xlsx
   💾 Checkpoint: 300/1000
✅ Збережено: mountains_1000_unique_enriched.xlsx
   💾 Checkpoint: 350/1000
✅ Збережено: mountains_1000_unique_enriched.xlsx
   💾 Checkpoint: 400/1000
✅ Збережено: mountains_1000_unique_enriched.xlsx
   💾 Checkpoint: 450/1000
✅ Збережено: mountains_1000_unique_enriched.xlsx
   💾 Checkpoint: 500/1000
✅ Збережено: mountains_1000_unique_enriched.xlsx
   💾 Checkpoint: 550/1000
✅ Збережено: mountains_1000_unique_enriched.xlsx
   💾 Checkpoint: 600/1000
✅ Збережено: mountains_1000_unique_enriched.xlsx
   💾 Checkpoint: 650/1000
✅ Збережено: mountains_100

,Mountain,Country,height,expected_height_m,deviation_percent
11,Romariswandköpfe,Austria,2934.0,3511,16.4
14,Hohes Aderl,Austria,2495.0,3506,28.8
22,Schneewinkelkopf,Austria,2396.0,3476,31.1
24,Ruderhofspitze,Austria,2494.0,3474,28.2
26,Hinterer Seelenkogel,Austria,3000.0,3469,13.5
28,Karlespitze,Austria,2740.0,3462,20.9
32,Hoher Zaun,Austria,329.0,3451,90.5
34,Collalto,Austria,1950.0,3436,43.2
35,Mittlerer Seelenkogl,Austria,2390.0,3426,30.2
36,Umbalköpfl,Austria,2396.0,3426,30.1


,rows,numeric_values,within_10_percent,accuracy_percent,api_error_rows,not_found_rows,needs_review_rows,llm_knowledge_rows,original_values_preserved,dataset,seconds
0,1000,976,572,57.2,23,1,109,743,True,mountains,1485.8


In [18]:
# Розкоментуй потрібні рядки, якщо хочеш завантажити результати.
# Завантаження можливе лише після виконання відповідного тесту.

# from google.colab import files
# files.download("capitals_1000_unique_enriched.xlsx")
# files.download("mountains_1000_unique_enriched.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---

## Висновки

### Що було реалізовано

Система автоматично доповнює Excel-файли через вебпошук і LLM, а для задачі прямої відстані між містами використовує детермінований розрахунок: онлайн-координати + haversine. Два тестових набори (столиці та гори) обробляються одним кодом, структура таблиць збережена.

### Технічний стек

| Компонент | Реалізація |
|-----------|------------|
| **LLM** | `gpt-4o-mini` (OpenAI) — дешева і достатньо точна для фактичних запитів |
| **Пошук** | Tavily API (`include_answer=True`) — повертає готову AI-відповідь + сніпети |
| **Structured output** | Pydantic v2 + `with_structured_output` — LLM повертає об'єкт, не рядок |
| **Паралельність** | `ThreadPoolExecutor(max_workers=8)` + `tqdm` — до 8 запитів одночасно |
| **Кеш** | `dict` за нормалізованим запитом — однакові запити не повторюються |
| **Retry** | Exponential backoff (2s → 4s), до 3 спроб — захист від rate-limit |
| **Distance engine** | Онлайн-координати міст (з країною) + haversine у Python |
| **Fallback** | Для не-distance задач: резервний запит → окремий LLM-промпт |
| **Вихідний файл** | `*_enriched.xlsx` з аркушами `data` і `enrichment_log` |
| **Масштабованість** | Checkpoint кожні 50 рядків, підтримка до 1000 записів |
| **Вартість** | `get_openai_callback` — реальний підрахунок токенів і вартості |

### Архітектурні рішення

**Чому pipeline, а не агент (LangGraph)?**  
Задача лінійна: читання → план → пошук → витягнення → запис. Граф вузлів додав би складності без користі. Pipeline простіше налагоджувати і легше пояснити.

**Чому два окремих промпти для Extractor і LLM-fallback?**  
Промпт `extract_value()` вимагає повертати `null`, якщо значення відсутнє в сніпетах — це правильна поведінка при роботі з реальними джерелами. `ask_llm_knowledge()` має інший системний промпт: явно дозволяє відповідати з тренувальних знань.

**Яку роль відіграє `cross_validate`?**  
Порівнює числове значення з вебпошуку і відповідь LLM з власних знань. Якщо розбіжність > 10%, рядок позначається `needs_review`, а деталь (`llm_check`, `diff`) записується в `note` для швидкої ручної перевірки.

### Обмеження

- Точність залежить від якості сніпетів Tavily і від того, наскільки добре LLM знає конкретну тему.
- `confidence` — самооцінка моделі, а не статистична ймовірність: модель може помилятися з високим `confidence`.
- Система не зберігає форматування оригінального Excel-файлу (стилі, формули, merged cells) — лише значення.